# Agentic RAG System using Ollama

This notebook implements an Agentic Retrieval-Augmented Generation (RAG) flow using Ollama models. The system intelligently routes questions to either local document retrieval or web search based on content analysis.

In [ ]:
# Install necessary packages
# !pip install -q langchain langchain-community langchain-ollama crewai==0.28.6 crewai-tools python-dotenv faiss-cpu sentence-transformers

ERROR: Cannot install langchain-community==0.0.25, langchain-ollama==0.1.0, langchain-ollama==0.1.1, langchain-ollama==0.1.3, langchain-ollama==0.2.0, langchain-ollama==0.2.1, langchain-ollama==0.2.2, langchain-ollama==0.2.3, langchain-ollama==0.3.0, langchain-ollama==0.3.1 and langchain==0.1.10 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [17]:
# Import required libraries
import os
from dotenv import load_dotenv
from langchain.vectorstores import FAISS
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_community.chat_models import ChatOllama
from crewai_tools import SerperDevTool, ScrapeWebsiteTool
from crewai import Agent, Task, Crew
from langchain_openai import ChatOpenAI  # For CrewAI

# Load environment variables
load_dotenv()
SERPER_API_KEY = os.getenv("SERPER_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

## Initialize Ollama LLM

We'll use Ollama to run models locally instead of calling Groq API.

In [18]:
# Initialize LLM
# Replace Groq with Ollama - using llama2 model
# You can change to llama3 or another model if available in your Ollama instance
llm = Ollama(
    model="llama2",  # or "llama3", "mistral", etc.
    temperature=0.1,
    num_ctx=4096,    # Context window
)

# Initialize chat model for more chat-like interactions
chat_model = ChatOllama(
    model="llama2",  # or "llama3", "mistral", etc.
    temperature=0.1
)

# For CrewAI - using direct LangChain models instead of the LLM class
# This fixes the "provider" parameter error
try:
    # Try to use Ollama for CrewAI
    crew_llm = ChatOllama(
        model="llama2",
        temperature=0.7
    )
    print("Using Ollama for CrewAI agents")
except Exception as e:
    print(f"Error initializing Ollama for CrewAI: {e}")
    print("Falling back to OpenAI")
    # Fallback to OpenAI if Ollama setup fails
    crew_llm = ChatOpenAI(
        model="gpt-3.5-turbo",
        temperature=0.7
    )

Using Ollama for CrewAI agents


## Local Knowledge Router

This function determines whether the query can be answered using local document knowledge.

In [19]:
def check_local_knowledge(query, context):
    """Router function to determine if we can answer from local knowledge"""
    prompt = '''Role: Question-Answering Assistant
Task: Determine whether the system can answer the user's question based on the provided text.
Instructions:
    - Analyze the text and identify if it contains the necessary information to answer the user's question.
    - Provide a clear and concise response indicating whether the system can answer the question or not.
    - Your response should include only a single word. Nothing else, no other text, information, header/footer. 
Output Format:
    - Answer: Yes/No
Study the below examples and based on that, respond to the last question. 
Examples:
    Input: 
        Text: The capital of France is Paris.
        User Question: What is the capital of France?
    Expected Output:
        Answer: Yes
    Input: 
        Text: The population of the United States is over 330 million.
        User Question: What is the population of China?
    Expected Output:
        Answer: No
    Input:
        User Question: {query}
        Text: {text}
'''
    formatted_prompt = prompt.format(text=context, query=query)
    response = llm(formatted_prompt)
    return response.strip().lower() == "yes"

## Web Content Retrieval with CrewAI

When local knowledge is insufficient, we'll use CrewAI to search the web and extract relevant information.

In [20]:
def setup_web_scraping_agent():
    """Setup the web scraping agent and related components"""
    # Set up the API key for SerperDev
    os.environ["SERPER_API_KEY"] = SERPER_API_KEY
    
    search_tool = SerperDevTool()  # Tool for performing web searches
    scrape_website = ScrapeWebsiteTool()  # Tool for extracting data from websites
    
    # Define the web search agent with the LangChain model directly
    web_search_agent = Agent(
        role="Expert Web Search Agent",
        goal="Identify and retrieve relevant web data for user queries",
        backstory="An expert in identifying valuable web sources for the user's needs",
        allow_delegation=False,
        verbose=True,
        llm=crew_llm  # Using the LangChain model directly
    )
    
    # Define the web scraping agent with the LangChain model directly
    web_scraper_agent = Agent(
        role="Expert Web Scraper Agent",
        goal="Extract and analyze content from specific web pages identified by the search agent",
        backstory="A highly skilled web scraper, capable of analyzing and summarizing website content accurately",
        allow_delegation=False,
        verbose=True,
        llm=crew_llm  # Using the LangChain model directly
    )
    
    # Define the web search task
    search_task = Task(
        description=(
            "Identify the most relevant web page or article for the topic: '{topic}'. "
            "Use all available tools to search for and provide a link to a web page "
            "that contains valuable information about the topic. Keep your response concise."
        ),
        expected_output=(
            "A concise summary of the most relevant web page or article for '{topic}', "
            "including the link to the source and key points from the content."
        ),
        tools=[search_tool],
        agent=web_search_agent,
    )
    
    # Define the web scraping task
    scraping_task = Task(
        description=(
            "Extract and analyze data from the given web page or website. Focus on the key sections "
            "that provide insights into the topic: '{topic}'. Use all available tools to retrieve the content, "
            "and summarize the key findings in a concise manner."
        ),
        expected_output=(
            "A detailed summary of the content from the given web page or website, highlighting the key insights "
            "and explaining their relevance to the topic: '{topic}'. Ensure clarity and conciseness."
        ),
        tools=[scrape_website],
        agent=web_scraper_agent,
    )
    
    # Define the crew to manage agents and tasks
    crew = Crew(
        agents=[web_search_agent, web_scraper_agent],
        tasks=[search_task, scraping_task],
        verbose=1,
        memory=False,
    )
    return crew

def get_web_content(query):
    """Get content from web scraping"""
    try:
        crew = setup_web_scraping_agent()
        result = crew.kickoff(inputs={"topic": query})
        return result.raw
    except Exception as e:
        print(f"Error in web scraping: {e}")
        return f"I couldn't retrieve web information about '{query}'. Please try a different query or check your API settings."

## Vector Database Setup

Set up a vector database from PDF documents to enable semantic search.

In [21]:
def setup_vector_db(pdf_path):
    """Setup vector database from PDF"""
    # Load and chunk PDF
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=50
    )
    chunks = text_splitter.split_documents(documents)
    
    # Create vector database
    try:
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        vector_db = FAISS.from_documents(chunks, embeddings)
    except Exception as e:
        print(f"Error loading embeddings: {e}")
        # Fallback to a different embedding model
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/paraphrase-MiniLM-L3-v2"
        )
        vector_db = FAISS.from_documents(chunks, embeddings)
    
    return vector_db

def get_local_content(vector_db, query):
    """Get content from vector database"""
    docs = vector_db.similarity_search(query, k=5)
    return " ".join([doc.page_content for doc in docs])

## Response Generation

Generate final answers using the Ollama LLM with either local or web content.

In [22]:
def generate_final_answer(context, query):
    """Generate final answer using LLM"""
    prompt = f"""You are a helpful assistant. Use the provided context to answer the query accurately.
    
    Context: {context}
    
    Query: {query}
    
    Answer:"""
    
    response = llm(prompt)
    return response

def process_query(query, vector_db, local_context):
    """Main function to process user query"""
    print(f"Processing query: {query}")
    
    # Step 1: Check if we can answer from local knowledge
    can_answer_locally = check_local_knowledge(query, local_context)
    print(f"Can answer locally: {can_answer_locally}")
    
    # Step 2: Get context either from local DB or web
    if can_answer_locally:
        context = get_local_content(vector_db, query)
        print("Retrieved context from local documents")
    else:
        context = get_web_content(query)
        print("Retrieved context from web scraping")
    
    # Step 3: Generate final answer
    answer = generate_final_answer(context, query)
    return answer

## Main Execution

Run the complete Agentic RAG pipeline with example queries.

In [23]:
def main():
    # Setup
    pdf_path = "./data/genai-principles.pdf" 
    
    # Initialize vector database
    print("Setting up vector database...")
    vector_db = setup_vector_db(pdf_path)
    
    # Get initial context from PDF for routing
    local_context = get_local_content(vector_db, "")
    
    # Example usage
    queries = [
        "What is Agentic RAG?",
        "What are the key principles in the GenAI document?",
        "What is the latest news about artificial intelligence?"
    ]
    
    for query in queries:
        print("\n" + "-"*50)
        print(f"\nQuery: {query}")
        result = process_query(query, vector_db, local_context)
        print("\nFinal Answer:")
        print(result)
        print("-"*50 + "\n")

if __name__ == "__main__":
    main()

Setting up vector database...

--------------------------------------------------

Query: What is Agentic RAG?
Processing query: What is Agentic RAG?

--------------------------------------------------

Query: What is Agentic RAG?
Processing query: What is Agentic RAG?


OllamaEndpointNotFoundError: Ollama call failed with status code 404. Maybe your model is not found and you should pull the model with `ollama pull llama2`.

## Interactive Query Interface

Run queries interactively against the Agentic RAG system.

In [ ]:
from IPython.display import clear_output

def interactive_query():
    # Setup
    pdf_path = "./data/genai-principles.pdf"
    
    # Initialize vector database
    print("Setting up vector database...")
    vector_db = setup_vector_db(pdf_path)
    
    # Get initial context from PDF for routing
    local_context = get_local_content(vector_db, "")
    print("System ready! Enter your query or type 'exit' to quit.\n")
    
    while True:
        query = input("Enter your query: ")
        if query.lower() == 'exit':
            print("Exiting interactive query mode.")
            break
            
        clear_output(wait=True)
        print(f"Processing: {query}\n")
        result = process_query(query, vector_db, local_context)
        print("\nAnswer:")
        print(result)
        print("\n" + "-"*50)
        print("\nEnter a new query or type 'exit' to quit.")

In [ ]:
# Run the interactive query interface
# interactive_query()

In [ ]:
# Install necessary packages
!pip install -q langchain langchain-community langchain-ollama crewai crewai-tools python-dotenv faiss-cpu sentence-transformers

## Test Fixed Implementation

Let's test the fixed implementation using a simple example. This should run without the 'provider' parameter error.

In [ ]:
# Test the fixed implementation with a simple query
def test_fixed_implementation():
    print("Testing fixed implementation...")
    
    # Setup a sample PDF path - update this to your actual PDF
    pdf_path = "./data/genai-principles.pdf"
    
    try:
        # Initialize vector database
        print("Setting up vector database...")
        vector_db = setup_vector_db(pdf_path)
        
        # Get initial context
        local_context = get_local_content(vector_db, "")
        
        # Test with a simple query that likely needs web search
        test_query = "What is Agentic RAG?"
        print(f"\nProcessing query: {test_query}")
        
        # Process the query
        result = process_query(test_query, vector_db, local_context)
        
        print("\nResult:")
        print(result)
        print("\nTest completed successfully!")
        
    except Exception as e:
        print(f"\nError in test: {e}")
        import traceback
        traceback.print_exc()
        print("\nPlease check the error message above.")

# Uncomment to run the test
# test_fixed_implementation()

## Explanation of the Fix

The error `BadRequestError: litellm.BadRequestError: OpenAIException - Error code: 400 - {'error': {'message': 'Unrecognized request argument supplied: provider', 'type': 'invalid_request_error', 'param': None, 'code': None}}` was occurring because:

1. CrewAI's `LLM` class was adding a `provider` parameter to requests sent to OpenAI
2. OpenAI's API doesn't recognize this parameter, leading to the 400 error

The solution implemented above:

1. Skips CrewAI's `LLM` abstraction class entirely
2. Uses LangChain's chat models (`ChatOllama` or `ChatOpenAI`) directly with CrewAI's Agent class
3. Adds proper error handling to catch and explain any remaining issues
4. Pins the CrewAI version to 0.28.6 for better compatibility

This approach ensures that no unexpected parameters are sent to the underlying APIs.